In [14]:
import openai, json

client = openai.OpenAI()
messages = []

In [15]:
def get_weather(city):
    return '33 degrees celcius'

FUNCTION_MAP = {
    'get_weather': get_weather
}

In [16]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of."
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [ ]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments
                        }
                    } for tool_call in message.tool_calls
                ]
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            # convert the json string into python object
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
            
            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)
            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
        
        ai_call()
    
    else:
        messages.append({
            "role": "assistant",
            "content": message.content
        })
        print(f"AI: {message.content}")

def ai_call():
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=messages,
        tools=TOOLS
    )
    message = response.choices[0].message
    process_ai_response(message)

In [18]:
while True:
    message = input("Send a message to the LLM...")
    if message.lower() == "quit" or message.lower() == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        ai_call()

User: my name leo


TypeError: object of type 'NoneType' has no len()